In [ ]:
# Import semua libraries yang dibutuhkan
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils import shuffle
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
import matplotlib.pyplot as plt
from google.colab import files
import warnings
import gc
import os

# Setup untuk optimasi memory
os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings('ignore')

print("✅ Setup completed!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print("🧹 Clean dataset ready for processing!")

✅ Setup completed!
PyTorch version: 2.6.0+cu124
CUDA available: False
🧹 Clean dataset ready for processing!


In [ ]:
# Upload dan load clean dataset
print("📁 Upload your clean dataset (clean_news_dataset.csv)")
uploaded = files.upload()

# Load dataset
df = pd.read_csv('clean_news_dataset.csv')

print("📊 DATASET OVERVIEW")
print("=" * 50)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")

# Display FULL dataset
print(f"\n📋 COMPLETE DATASET (All {len(df)} rows):")
print("=" * 70)
display(df)  # Menampilkan semua data dalam bentuk dataframe

# Basic info
print(f"\n📈 DATASET STATISTICS:")
print(df.info())
print(f"\nMissing values:")
print(df.isnull().sum())

📁 Upload your clean dataset (clean_news_dataset.csv)


Saving clean_news_dataset.csv to clean_news_dataset.csv
📊 DATASET OVERVIEW
Shape: (248, 5)
Columns: ['judul', 'label', 'sumber', 'kategori', 'title_length']
Memory usage: 77.5 KB

📋 COMPLETE DATASET (All 248 rows):


,judul,label,sumber,kategori,title_length
0,SALAH Video Pengakuan Kekalahan Israel,FAKE,TurnBackHoax.id,News,38
1,SALAH Video Warga Israel Mengungsi di Gurun Pasir,FAKE,TurnBackHoax.id,News,49
2,SALAH Artikel CNBC Indonesia: Raja Salman Sebu...,FAKE,TurnBackHoax.id,News,87
3,PENIPUAN Tautan Pendaftaran CPNS Kemenhub,FAKE,TurnBackHoax.id,News,41
4,SALAH Video Monster Drone Buatan Iran,FAKE,TurnBackHoax.id,News,37
...,...,...,...,...,...
243,Airlangga dorong pengembangan investasi Temase...,REAL,Antara News,News,69
244,Vale (INCO) jadwalkan ulang RUPSLB penunjukan ...,REAL,Antara News,News,58
245,Yogyakarta siapkan lokasi servis becak listrik...,REAL,Antara News,News,59
246,Perlu upaya serius kurangi ketergantungan impo...,REAL,Antara News,News,57



📈 DATASET STATISTICS:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 248 entries, 0 to 247
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   judul         248 non-null    object
 1   label         248 non-null    object
 2   sumber        248 non-null    object
 3   kategori      248 non-null    object
 4   title_length  248 non-null    int64 
dtypes: int64(1), object(4)
memory usage: 9.8+ KB
None

Missing values:
judul           0
label           0
sumber          0
kategori        0
title_length    0
dtype: int64


In [ ]:
# Analisis distribusi label
label_counts = df['label'].value_counts()
label_percentages = df['label'].value_counts(normalize=True) * 100

print("🏷️ LABEL DISTRIBUTION ANALYSIS")
print("=" * 40)
print(f"FAKE: {label_counts['FAKE']} articles ({label_percentages['FAKE']:.1f}%)")
print(f"REAL: {label_counts['REAL']} articles ({label_percentages['REAL']:.1f}%)")
print(f"Total: {len(df)} articles")

# Interactive Pie Chart
fig = px.pie(
    values=label_counts.values,
    names=label_counts.index,
    title="📊 Distribusi Label Berita (Interactive)",
    color_discrete_map={'REAL': '#27AE60', 'FAKE': '#E74C3C'},
    hole=0.4,
    hover_data=[label_counts.values]
)

fig.update_traces(
    textposition='inside',
    textinfo='percent+label',
    hovertemplate='<b>%{label}</b><br>Count: %{value}<br>Percentage: %{percent}<extra></extra>'
)

fig.update_layout(
    font_size=14,
    showlegend=True,
    width=700,
    height=500,
    title_x=0.5
)

fig.show()

# Display label distribution sebagai dataframe
label_df = pd.DataFrame({
    'Label': label_counts.index,
    'Count': label_counts.values,
    'Percentage': label_percentages.values.round(1)
})

print(f"\n📋 Label Distribution DataFrame:")
display(label_df)

🏷️ LABEL DISTRIBUTION ANALYSIS
FAKE: 98 articles (39.5%)
REAL: 150 articles (60.5%)
Total: 248 articles



📋 Label Distribution DataFrame:


,Label,Count,Percentage
0,REAL,150,60.5
1,FAKE,98,39.5


In [ ]:
# Analisis distribusi sumber
source_counts = df['sumber'].value_counts()
source_percentages = df['sumber'].value_counts(normalize=True) * 100

print("📰 SOURCE DISTRIBUTION ANALYSIS")
print("=" * 40)

# Interactive Bar Chart
fig = px.bar(
    x=source_counts.index,
    y=source_counts.values,
    title="📰 Distribusi Sumber Berita (Interactive)",
    labels={'x': 'Sumber', 'y': 'Jumlah Artikel'},
    color=source_counts.values,
    color_continuous_scale='viridis',
    text=source_counts.values
)

fig.update_traces(
    texttemplate='%{text}',
    textposition='outside',
    hovertemplate='<b>%{x}</b><br>Articles: %{y}<extra></extra>'
)

fig.update_layout(
    showlegend=False,
    width=800,
    height=500,
    title_x=0.5,
    xaxis_title="Sumber Berita",
    yaxis_title="Jumlah Artikel"
)

fig.show()

# Display source distribution sebagai dataframe
source_df = pd.DataFrame({
    'Sumber': source_counts.index,
    'Count': source_counts.values,
    'Percentage': source_percentages.values.round(1)
})

print(f"\n📋 Source Distribution DataFrame:")
display(source_df)

# Cross-tabulation: Source vs Label
crosstab = pd.crosstab(df['sumber'], df['label'], margins=True)
print(f"\n📊 Cross-tabulation: Source vs Label")
display(crosstab)

📰 SOURCE DISTRIBUTION ANALYSIS



📋 Source Distribution DataFrame:


,Sumber,Count,Percentage
0,Kompas.com,100,40.3
1,TurnBackHoax.id,98,39.5
2,CNN Indonesia,25,10.1
3,Antara News,25,10.1



📊 Cross-tabulation: Source vs Label


label,FAKE,REAL,All
sumber,,,
Antara News,0,25,25
CNN Indonesia,0,25,25
Kompas.com,0,100,100
TurnBackHoax.id,98,0,98
All,98,150,248


In [ ]:
# Analisis panjang judul
print("📏 TITLE LENGTH ANALYSIS")
print("=" * 40)

# Statistical summary
title_stats = df['title_length'].describe()
print("Title Length Statistics:")
print(title_stats)

# Title length by label
fake_lengths = df[df['label'] == 'FAKE']['title_length']
real_lengths = df[df['label'] == 'REAL']['title_length']

print(f"\nFAKE news title length: Mean={fake_lengths.mean():.1f}, Std={fake_lengths.std():.1f}")
print(f"REAL news title length: Mean={real_lengths.mean():.1f}, Std={real_lengths.std():.1f}")

# Interactive Box Plot
fig = px.box(
    df,
    x='label',
    y='title_length',
    title="📏 Distribusi Panjang Judul berdasarkan Label (Interactive)",
    color='label',
    color_discrete_map={'REAL': '#27AE60', 'FAKE': '#E74C3C'},
    points="outliers"
)

fig.update_traces(
    hovertemplate='<b>%{x}</b><br>Length: %{y} characters<extra></extra>'
)

fig.update_layout(
    width=700,
    height=500,
    title_x=0.5,
    xaxis_title="Label",
    yaxis_title="Panjang Judul (karakter)"
)

fig.show()

# Interactive Histogram
fig2 = px.histogram(
    df,
    x='title_length',
    color='label',
    title="📊 Histogram Panjang Judul (Interactive)",
    color_discrete_map={'REAL': '#27AE60', 'FAKE': '#E74C3C'},
    opacity=0.7,
    marginal="box",
    nbins=20
)

fig2.update_layout(
    width=800,
    height=500,
    title_x=0.5,
    xaxis_title="Panjang Judul (karakter)",
    yaxis_title="Frekuensi"
)

fig2.show()

# Display title length stats sebagai dataframe
title_stats_df = pd.DataFrame({
    'Label': ['FAKE', 'REAL', 'Overall'],
    'Mean': [fake_lengths.mean(), real_lengths.mean(), df['title_length'].mean()],
    'Std': [fake_lengths.std(), real_lengths.std(), df['title_length'].std()],
    'Min': [fake_lengths.min(), real_lengths.min(), df['title_length'].min()],
    'Max': [fake_lengths.max(), real_lengths.max(), df['title_length'].max()]
}).round(2)

print(f"\n📋 Title Length Statistics DataFrame:")
display(title_stats_df)

📏 TITLE LENGTH ANALYSIS
Title Length Statistics:
count    248.000000
mean      62.455645
std       14.170379
min       25.000000
25%       51.750000
50%       64.000000
75%       71.000000
max      100.000000
Name: title_length, dtype: float64

FAKE news title length: Mean=56.3, Std=13.9
REAL news title length: Mean=66.5, Std=12.8



📋 Title Length Statistics DataFrame:


,Label,Mean,Std,Min,Max
0,FAKE,56.27,13.95,28,95
1,REAL,66.50,12.83,25,100
2,Overall,62.46,14.17,25,100


In [ ]:
# Analisis sample data untuk setiap kategori
print("📝 SAMPLE DATA ANALYSIS")
print("=" * 50)

# Sample FAKE news
print("❌ FAKE NEWS SAMPLES:")
fake_samples = df[df['label'] == 'FAKE'].head(5)
display(fake_samples)

print(f"\n✅ REAL NEWS SAMPLES:")
real_samples = df[df['label'] == 'REAL'].head(5)
display(real_samples)

# Analisis kata-kata umum di judul FAKE vs REAL
print(f"\n🔍 PATTERN ANALYSIS:")

# Common words in FAKE news titles
fake_titles = ' '.join(df[df['label'] == 'FAKE']['judul'].astype(str))
real_titles = ' '.join(df[df['label'] == 'REAL']['judul'].astype(str))

# Simple word frequency (top 10)
from collections import Counter
import re

def get_common_words(text, n=10):
    # Simple preprocessing
    words = re.findall(r'\b[A-Za-z]+\b', text.lower())
    # Filter out common stop words
    stop_words = {'dan', 'di', 'ke', 'dari', 'yang', 'untuk', 'dengan', 'pada', 'adalah', 'ini', 'itu', 'akan', 'telah', 'atau', 'juga', 'tidak', 'dalam', 'oleh', 'dapat', 'sudah', 'seperti', 'antara', 'atas', 'bisa', 'sama', 'satu', 'dua', 'tiga', 'the', 'and', 'of', 'to', 'in', 'for', 'is', 'it', 'on', 'as', 'be', 'at', 'by', 'this', 'have', 'from', 'they', 'she', 'her', 'been', 'than', 'its', 'his', 'had', 'has'}
    filtered_words = [word for word in words if word not in stop_words and len(word) > 3]
    return Counter(filtered_words).most_common(n)

fake_common = get_common_words(fake_titles)
real_common = get_common_words(real_titles)

fake_words_df = pd.DataFrame(fake_common, columns=['Word', 'Frequency'])
fake_words_df['Type'] = 'FAKE'

real_words_df = pd.DataFrame(real_common, columns=['Word', 'Frequency'])
real_words_df['Type'] = 'REAL'

print("📊 Top 10 Words in FAKE News Titles:")
display(fake_words_df)

print("📊 Top 10 Words in REAL News Titles:")
display(real_words_df)

📝 SAMPLE DATA ANALYSIS
❌ FAKE NEWS SAMPLES:


,judul,label,sumber,kategori,title_length
0,SALAH Video Pengakuan Kekalahan Israel,FAKE,TurnBackHoax.id,News,38
1,SALAH Video Warga Israel Mengungsi di Gurun Pasir,FAKE,TurnBackHoax.id,News,49
2,SALAH Artikel CNBC Indonesia: Raja Salman Sebu...,FAKE,TurnBackHoax.id,News,87
3,PENIPUAN Tautan Pendaftaran CPNS Kemenhub,FAKE,TurnBackHoax.id,News,41
4,SALAH Video Monster Drone Buatan Iran,FAKE,TurnBackHoax.id,News,37



✅ REAL NEWS SAMPLES:


,judul,label,sumber,kategori,title_length
98,"Ratusan UMKM Go Digital, Ini Perjalanan Kiki A...",REAL,Kompas.com,News,76
99,"Kenalkan Reverse Engineering di Jambore IRCC, ...",REAL,Kompas.com,News,89
100,ASEAN U-23 Mandiri Cup 2025: Kalahkan Filipina...,REAL,Kompas.com,News,89
101,VIKVisualInteraktifKompas,REAL,Kompas.com,News,25
102,"2 Promo Liburan ke Jepang, Tiket Pesawat Murah...",REAL,Kompas.com,News,79



🔍 PATTERN ANALYSIS:
📊 Top 10 Words in FAKE News Titles:


,Word,Frequency,Type
0,salah,81,FAKE
1,israel,27,FAKE
2,iran,22,FAKE
3,video,21,FAKE
4,indonesia,18,FAKE
5,penipuan,17,FAKE
6,tautan,9,FAKE
7,pendaftaran,9,FAKE
8,karena,9,FAKE
9,perang,7,FAKE


📊 Top 10 Words in REAL News Titles:


,Word,Frequency,Type
0,kebakaran,9,REAL
1,tebet,9,REAL
2,jadi,9,REAL
3,indonesia,8,REAL
4,trump,8,REAL
5,rumah,7,REAL
6,tewas,7,REAL
7,anak,7,REAL
8,kongres,6,REAL
9,cerita,5,REAL


In [ ]:
# Data balancing dan shuffling untuk training
print("⚖️ DATA BALANCING & SHUFFLING")
print("=" * 50)

# Check current balance
current_balance = df['label'].value_counts()
fake_count = current_balance.get('FAKE', 0)
real_count = current_balance.get('REAL', 0)

print(f"📊 Current Distribution:")
print(f"   FAKE: {fake_count} articles ({fake_count/len(df)*100:.1f}%)")
print(f"   REAL: {real_count} articles ({real_count/len(df)*100:.1f}%)")
print(f"   Difference: {abs(fake_count - real_count)} articles")

# Balance dataset - ambil jumlah yang lebih sedikit
min_count = min(fake_count, real_count)
print(f"\n🎯 Balancing to {min_count} articles per class...")

# Sample equal amounts from each class
fake_balanced = df[df['label'] == 'FAKE'].sample(n=min_count, random_state=42)
real_balanced = df[df['label'] == 'REAL'].sample(n=min_count, random_state=42)

# Combine and shuffle
df_balanced = pd.concat([fake_balanced, real_balanced], ignore_index=True)
df_balanced = shuffle(df_balanced, random_state=42).reset_index(drop=True)

print(f"✅ Balanced Dataset:")
print(f"   Total: {len(df_balanced)} articles")
print(f"   FAKE: {len(fake_balanced)} articles (50.0%)")
print(f"   REAL: {len(real_balanced)} articles (50.0%)")

# Display balanced dataset
print(f"\n📋 BALANCED & SHUFFLED DATASET:")
display(df_balanced)

# Verification - check randomness
print(f"\n🔀 Randomness Check (first 10 labels):")
print(df_balanced['label'].head(10).tolist())

# Save balanced dataset
df_balanced.to_csv('balanced_dataset.csv', index=False)
print(f"\n💾 Balanced dataset saved as 'balanced_dataset.csv'")

⚖️ DATA BALANCING & SHUFFLING
📊 Current Distribution:
   FAKE: 98 articles (39.5%)
   REAL: 150 articles (60.5%)
   Difference: 52 articles

🎯 Balancing to 98 articles per class...
✅ Balanced Dataset:
   Total: 196 articles
   FAKE: 98 articles (50.0%)
   REAL: 98 articles (50.0%)

📋 BALANCED & SHUFFLED DATASET:


,judul,label,sumber,kategori,title_length
0,"Andai Hewan Peliharan Bisa Jadi Manusia, Ini H...",REAL,Kompas.com,News,67
1,Mengenal myBCA Keyboard: Cek Saldo hingga Tran...,REAL,Kompas.com,News,70
2,SALAH Iran Pamerkan Rudal Terbesar di Dunia,FAKE,TurnBackHoax.id,News,43
3,PENIPUAN Tautan Pendaftaran Lowongan Kerja Sup...,FAKE,TurnBackHoax.id,News,52
4,"Identitas 4 Korban Tewas Kebakaran di Tebet, S...",REAL,Kompas.com,News,64
...,...,...,...,...,...
191,4 Fitur AI yang Bisa Diakses Langsung dari Cov...,REAL,Kompas.com,News,79
192,SALAH Potret Prabowo dan Perwakilan Belanda Ba...,FAKE,TurnBackHoax.id,News,72
193,PENIPUAN Tautan Pendaftaran Bantuan Subsidi Upah,FAKE,TurnBackHoax.id,News,48
194,Menggaungkan Suara Lebih Lantang dari Negara-n...,REAL,Kompas.com,News,68



🔀 Randomness Check (first 10 labels):
['REAL', 'REAL', 'FAKE', 'FAKE', 'REAL', 'REAL', 'FAKE', 'FAKE', 'FAKE', 'REAL']

💾 Balanced dataset saved as 'balanced_dataset.csv'


In [ ]:
# Stratified train-test split
print("🎯 STRATIFIED TRAIN-TEST SPLIT")
print("=" * 50)

# Prepare features and target
X = df_balanced['judul'].values
y = df_balanced['label'].map({'FAKE': 0, 'REAL': 1}).values

print(f"📊 Preparing for split:")
print(f"   Total samples: {len(X)}")
print(f"   Features: News titles")
print(f"   Target: FAKE=0, REAL=1")

# Stratified split - mempertahankan proporsi label
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,          # 80% train, 20% test
    random_state=42,        # Reproducible results
    stratify=y              # 🎯 KEY: Maintain label proportion
)

# Convert back to dataframes for analysis
train_indices = df_balanced.index[:len(X_train)]
test_indices = df_balanced.index[len(X_train):]

# Create train/test dataframes
df_train = df_balanced.iloc[:len(X_train)].copy()
df_test = df_balanced.iloc[len(X_train):].copy()

# Actually use the correct stratified split
train_mask = df_balanced.index.isin(train_test_split(df_balanced.index, test_size=0.2, random_state=42, stratify=df_balanced['label'])[0])
df_train = df_balanced[train_mask].copy()
df_test = df_balanced[~train_mask].copy()

# Update X_train, X_test, y_train, y_test with correct split
X_train = df_train['judul'].values
X_test = df_test['judul'].values
y_train = df_train['label'].map({'FAKE': 0, 'REAL': 1}).values
y_test = df_test['label'].map({'FAKE': 0, 'REAL': 1}).values

print(f"✅ Split completed:")
print(f"   Training set: {len(X_train)} samples")
print(f"   Test set: {len(X_test)} samples")

# Verify stratification
train_fake_pct = (sum(y_train == 0) / len(y_train)) * 100
test_fake_pct = (sum(y_test == 0) / len(y_test)) * 100

print(f"\n📊 Stratification Verification:")
print(f"   Training FAKE: {sum(y_train == 0)}/{len(y_train)} ({train_fake_pct:.1f}%)")
print(f"   Training REAL: {sum(y_train == 1)}/{len(y_train)} ({100-train_fake_pct:.1f}%)")
print(f"   Test FAKE: {sum(y_test == 0)}/{len(y_test)} ({test_fake_pct:.1f}%)")
print(f"   Test REAL: {sum(y_test == 1)}/{len(y_test)} ({100-test_fake_pct:.1f}%)")
print(f"   Difference: {abs(train_fake_pct - test_fake_pct):.1f}% (should be <2%)")

if abs(train_fake_pct - test_fake_pct) < 2:
    print("✅ Perfect stratification achieved!")
else:
    print("⚠️ Stratification needs adjustment")

# Display train and test sets
print(f"\n📋 TRAINING SET ({len(df_train)} samples):")
display(df_train)

print(f"\n📋 TEST SET ({len(df_test)} samples):")
display(df_test)

🎯 STRATIFIED TRAIN-TEST SPLIT
📊 Preparing for split:
   Total samples: 196
   Features: News titles
   Target: FAKE=0, REAL=1
✅ Split completed:
   Training set: 156 samples
   Test set: 40 samples

📊 Stratification Verification:
   Training FAKE: 78/156 (50.0%)
   Training REAL: 78/156 (50.0%)
   Test FAKE: 20/40 (50.0%)
   Test REAL: 20/40 (50.0%)
   Difference: 0.0% (should be <2%)
✅ Perfect stratification achieved!

📋 TRAINING SET (156 samples):


,judul,label,sumber,kategori,title_length
0,"Andai Hewan Peliharan Bisa Jadi Manusia, Ini H...",REAL,Kompas.com,News,67
1,Mengenal myBCA Keyboard: Cek Saldo hingga Tran...,REAL,Kompas.com,News,70
2,SALAH Iran Pamerkan Rudal Terbesar di Dunia,FAKE,TurnBackHoax.id,News,43
4,"Identitas 4 Korban Tewas Kebakaran di Tebet, S...",REAL,Kompas.com,News,64
5,Cerita Jeffrie PSI Minta Ada Darah Jokowi agar...,REAL,Kompas.com,News,67
...,...,...,...,...,...
190,SALAH Ratusan Kapal Perang Cina Tiba di Timur ...,FAKE,TurnBackHoax.id,News,52
192,SALAH Potret Prabowo dan Perwakilan Belanda Ba...,FAKE,TurnBackHoax.id,News,72
193,PENIPUAN Tautan Pendaftaran Bantuan Subsidi Upah,FAKE,TurnBackHoax.id,News,48
194,Menggaungkan Suara Lebih Lantang dari Negara-n...,REAL,Kompas.com,News,68



📋 TEST SET (40 samples):


,judul,label,sumber,kategori,title_length
3,PENIPUAN Tautan Pendaftaran Lowongan Kerja Sup...,FAKE,TurnBackHoax.id,News,52
6,SALAH Megawati Menjerit karena Skandal Korupsi...,FAKE,TurnBackHoax.id,News,80
9,Kongres PSI 2025: Jeffrie Geovanie Bahas Cebon...,REAL,Kompas.com,News,59
12,"ASN DKI Wajib Olahraga Tiap Jumat, Rano Karno:...",REAL,Kompas.com,News,86
14,"Hasil Kamera Samsung Galaxy Z Flip 7: Detail, ...",REAL,Kompas.com,News,84
20,"Enggan Langsung Tinjau Kebakaran di Tebet, Ran...",REAL,Kompas.com,News,91
29,SALAH Rudal Indonesia Siap Dikirim ke Iran,FAKE,TurnBackHoax.id,News,42
39,SALAH Syarat Beli Rumah Subsidi: Punya Gaji Rp...,FAKE,TurnBackHoax.id,News,53
40,SALAH FIFA Blacklist Iran dan Tunjuk Indonesia...,FAKE,TurnBackHoax.id,News,75
43,SALAH Indonesia Menyatakan Perang Lawan Myanmar,FAKE,TurnBackHoax.id,News,47


In [ ]:
# Load pre-trained IndoBERT model untuk bahasa Indonesia
print("🤖 LOADING BERT MODEL & TOKENIZER")
print("=" * 50)

# Clear memory sebelum load model
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

# Pilih model yang optimal untuk bahasa Indonesia
model_name = "indolem/indobert-base-uncased"  # IndoBERT untuk bahasa Indonesia
# Alternative: "distilbert-base-uncased" (lebih ringan tapi English)

print(f"📥 Loading model: {model_name}")
print("⏳ This may take a few minutes...")

try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,  # Binary classification: FAKE vs REAL
        output_attentions=False,
        output_hidden_states=False,
    )

    print("✅ Model loaded successfully!")
    print(f"📊 Model info:")
    print(f"   Model: {model_name}")
    print(f"   Task: Binary Classification (FAKE vs REAL)")
    print(f"   Max sequence length: 512 tokens")
    print(f"   Vocabulary size: {tokenizer.vocab_size}")

except Exception as e:
    print(f"❌ Error loading IndoBERT. Falling back to DistilBERT...")
    model_name = "distilbert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2
    )
    print(f"✅ Fallback model loaded: {model_name}")

# Display model architecture summary
print(f"\n🏗️ Model Architecture:")
print(f"   Parameters: ~{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"   Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

🤖 LOADING BERT MODEL & TOKENIZER
📥 Loading model: indolem/indobert-base-uncased
⏳ This may take a few minutes...


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/445M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indolem/indobert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Model loaded successfully!
📊 Model info:
   Model: indolem/indobert-base-uncased
   Task: Binary Classification (FAKE vs REAL)
   Max sequence length: 512 tokens
   Vocabulary size: 31923

🏗️ Model Architecture:
   Parameters: ~110.6M
   Device: CPU


In [ ]:
# Tokenize data dengan optimasi memory
print("🔤 TOKENIZING DATA")
print("=" * 30)

def tokenize_texts(texts, max_length=128):
    """
    Tokenize texts dengan memory optimization
    """
    print(f"Tokenizing {len(texts)} texts...")

    encoded = tokenizer(
        list(texts),
        truncation=True,      # Potong jika lebih dari max_length
        padding=True,         # Pad hingga max_length
        max_length=max_length, # 128 tokens (lebih pendek = lebih hemat memory)
        return_tensors="pt"   # Return PyTorch tensors
    )

    return encoded

# Tokenize training dan test data
print("🔄 Tokenizing training data...")
train_encodings = tokenize_texts(X_train, max_length=128)

print("🔄 Tokenizing test data...")
test_encodings = tokenize_texts(X_test, max_length=128)

print("✅ Tokenization completed!")
print(f"📊 Tokenization summary:")
print(f"   Max length: 128 tokens (memory optimized)")
print(f"   Training samples: {len(X_train)}")
print(f"   Test samples: {len(X_test)}")
print(f"   Encoding keys: {list(train_encodings.keys())}")

# Memory usage check
train_memory = sum(tensor.nelement() * tensor.element_size() for tensor in train_encodings.values()) / 1024 / 1024
test_memory = sum(tensor.nelement() * tensor.element_size() for tensor in test_encodings.values()) / 1024 / 1024

print(f"   Training encodings memory: {train_memory:.2f} MB")
print(f"   Test encodings memory: {test_memory:.2f} MB")
print(f"   Total tokenization memory: {train_memory + test_memory:.2f} MB")

# Show sample tokenization
print(f"\n📝 Sample tokenization:")
sample_text = X_train[0]
sample_tokens = tokenizer.tokenize(sample_text)
print(f"   Original: {sample_text[:100]}...")
print(f"   Tokens ({len(sample_tokens)}): {sample_tokens[:15]}...")

🔤 TOKENIZING DATA
🔄 Tokenizing training data...
Tokenizing 156 texts...
🔄 Tokenizing test data...
Tokenizing 40 texts...
✅ Tokenization completed!
📊 Tokenization summary:
   Max length: 128 tokens (memory optimized)
   Training samples: 156
   Test samples: 40
   Encoding keys: ['input_ids', 'token_type_ids', 'attention_mask']
   Training encodings memory: 0.09 MB
   Test encodings memory: 0.02 MB
   Total tokenization memory: 0.11 MB

📝 Sample tokenization:
   Original: Andai Hewan Peliharan Bisa Jadi Manusia, Ini Harapan Pemilik Anabul...
   Tokens (13): ['andai', 'hewan', 'pelihara', '##n', 'bisa', 'jadi', 'manusia', ',', 'ini', 'harapan', 'pemilik', 'ana', '##bul']...


In [ ]:
# Custom dataset class untuk PyTorch training
print("📦 CREATING PYTORCH DATASETS")
print("=" * 40)

class FakeNewsDataset(torch.utils.data.Dataset):
    """
    Custom Dataset class untuk BERT fake news classification
    """
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        # Get encoded inputs
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        # Add labels
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

# Create datasets
print("🔨 Creating training dataset...")
train_dataset = FakeNewsDataset(train_encodings, y_train)

print("🔨 Creating test dataset...")
test_dataset = FakeNewsDataset(test_encodings, y_test)

print("✅ Datasets created successfully!")
print(f"📊 Dataset info:")
print(f"   Training dataset size: {len(train_dataset)}")
print(f"   Test dataset size: {len(test_dataset)}")

# Test dataset functionality
print(f"\n🧪 Testing dataset functionality:")
sample_item = train_dataset[0]
print(f"   Sample item keys: {list(sample_item.keys())}")
print(f"   Input shape: {sample_item['input_ids'].shape}")
print(f"   Attention mask shape: {sample_item['attention_mask'].shape}")
print(f"   Label: {sample_item['labels'].item()} ({'FAKE' if sample_item['labels'].item() == 0 else 'REAL'})")

# Memory cleanup
gc.collect()
print(f"   Memory cleanup completed")

📦 CREATING PYTORCH DATASETS
🔨 Creating training dataset...
🔨 Creating test dataset...
✅ Datasets created successfully!
📊 Dataset info:
   Training dataset size: 156
   Test dataset size: 40

🧪 Testing dataset functionality:
   Sample item keys: ['input_ids', 'token_type_ids', 'attention_mask', 'labels']
   Input shape: torch.Size([26])
   Attention mask shape: torch.Size([26])
   Label: 1 (REAL)
   Memory cleanup completed


In [ ]:
# Setup training arguments dengan konfigurasi ringan
print("⚙️ TRAINING CONFIGURATION")
print("=" * 40)

# Create output directory
import os
os.makedirs('./results', exist_ok=True)

# Ultra-light training configuration untuk Colab
training_args = TrainingArguments(
    # Output and logging
    output_dir='./results',
    logging_dir='./logs',
    logging_steps=5,

    # Training parameters
    num_train_epochs=3,              # Hanya 3 epochs (cepat)
    per_device_train_batch_size=4,   # Batch size sangat kecil
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,   # Simulasi batch size 8

    # Optimization
    learning_rate=2e-5,              # Standard BERT learning rate
    weight_decay=0.01,
    warmup_steps=10,

    # Evaluation and saving
    eval_strategy="epoch",           # Evaluate setiap epoch
    save_strategy="no",              # Tidak save model (hemat space)
    load_best_model_at_end=False,    # Tidak load best model

    # Memory optimization
    dataloader_pin_memory=False,     # Hemat memory
    remove_unused_columns=False,
    report_to=None,                  # Disable reporting ke wandb

    # Performance
    use_cpu=not torch.cuda.is_available(),  # Use CPU jika GPU tidak ada
)

print("✅ Training configuration completed!")
print(f"🎯 Training settings:")
print(f"   Epochs: {training_args.num_train_epochs}")
print(f"   Batch size: {training_args.per_device_train_batch_size}")
print(f"   Learning rate: {training_args.learning_rate}")
print(f"   Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"   Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"   Device: {'CUDA' if torch.cuda.is_available() and not training_args.use_cpu else 'CPU'}")

# Estimate training time
total_steps = (len(train_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)) * training_args.num_train_epochs
print(f"   Estimated steps: {total_steps}")
print(f"   Estimated time: 5-10 minutes")

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


⚙️ TRAINING CONFIGURATION
✅ Training configuration completed!
🎯 Training settings:
   Epochs: 3
   Batch size: 4
   Learning rate: 2e-05
   Gradient accumulation: 2
   Effective batch size: 8
   Device: CPU
   Estimated steps: 57
   Estimated time: 5-10 minutes


In [ ]:
# Initialize trainer dan mulai training
print("🚀 TRAINING BERT MODEL")
print("=" * 40)

# Clear memory sebelum training
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
)

print("🎯 Training started...")
print("⏰ This will take approximately 5-10 minutes")
print("📊 Progress will be shown below:")
print("-" * 50)

# Start training
try:
    training_output = trainer.train()

    print("\n" + "=" * 50)
    print("🎉 TRAINING COMPLETED SUCCESSFULLY!")
    print("=" * 50)

    # Display training results
    final_loss = training_output.training_loss
    print(f"📈 Training Results:")
    print(f"   Final training loss: {final_loss:.4f}")
    print(f"   Total training steps: {training_output.global_step}")
    print(f"   Training time: {training_output.metrics.get('train_runtime', 0):.1f} seconds")

except Exception as e:
    print(f"❌ Training failed: {str(e)}")
    print("💡 Try reducing batch size or switching to CPU")

# Memory cleanup after training
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print(f"🧹 Memory cleanup completed")

🚀 TRAINING BERT MODEL
🎯 Training started...
⏰ This will take approximately 5-10 minutes
📊 Progress will be shown below:
--------------------------------------------------


Epoch,Training Loss,Validation Loss
1,0.563000,0.587595
2,0.492800,0.388458
3,0.385900,0.366857



🎉 TRAINING COMPLETED SUCCESSFULLY!
📈 Training Results:
   Final training loss: 0.5498
   Total training steps: 60
   Training time: 249.2 seconds
🧹 Memory cleanup completed


In [ ]:
# Evaluasi model yang telah dilatih
print("📊 MODEL EVALUATION")
print("=" * 40)

# Evaluate on test set
print("🔍 Evaluating on test set...")
eval_results = trainer.evaluate(eval_dataset=test_dataset)

print(f"📈 Evaluation Results:")
for key, value in eval_results.items():
    if isinstance(value, float):
        print(f"   {key}: {value:.4f}")
    else:
        print(f"   {key}: {value}")

# Get predictions
print(f"\n🎯 Getting predictions...")
predictions_output = trainer.predict(test_dataset)
y_pred = np.argmax(predictions_output.predictions, axis=1)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
print(f"\n📊 Detailed Metrics:")
print(f"   Accuracy: {accuracy:.4f} ({accuracy*100:.1f}%)")

# Classification report
target_names = ['FAKE', 'REAL']
class_report = classification_report(y_test, y_pred, target_names=target_names, output_dict=True)

print(f"\n📋 Classification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))

# Create metrics dataframe
metrics_data = []
for label in ['FAKE', 'REAL']:
    metrics_data.append({
        'Class': label,
        'Precision': class_report[label]['precision'],
        'Recall': class_report[label]['recall'],
        'F1-Score': class_report[label]['f1-score'],
        'Support': class_report[label]['support']
    })

metrics_df = pd.DataFrame(metrics_data)
print(f"\n📋 Metrics DataFrame:")
display(metrics_df.round(4))

# Show macro and weighted averages
print(f"\n🎯 Overall Performance:")
print(f"   Macro avg F1: {class_report['macro avg']['f1-score']:.4f}")
print(f"   Weighted avg F1: {class_report['weighted avg']['f1-score']:.4f}")
print(f"   Accuracy: {class_report['accuracy']:.4f}")

📊 MODEL EVALUATION
🔍 Evaluating on test set...


📈 Evaluation Results:
   eval_loss: 0.3669
   eval_runtime: 7.5021
   eval_samples_per_second: 5.3320
   eval_steps_per_second: 1.3330
   epoch: 3.0000

🎯 Getting predictions...

📊 Detailed Metrics:
   Accuracy: 0.8500 (85.0%)

📋 Classification Report:
              precision    recall  f1-score   support

        FAKE       0.94      0.75      0.83        20
        REAL       0.79      0.95      0.86        20

    accuracy                           0.85        40
   macro avg       0.86      0.85      0.85        40
weighted avg       0.86      0.85      0.85        40


📋 Metrics DataFrame:


,Class,Precision,Recall,F1-Score,Support
0,FAKE,0.9375,0.75,0.8333,20.0
1,REAL,0.7917,0.95,0.8636,20.0



🎯 Overall Performance:
   Macro avg F1: 0.8485
   Weighted avg F1: 0.8485
   Accuracy: 0.8500


In [ ]:
# Visualisasi confusion matrix
print("🔍 CONFUSION MATRIX VISUALIZATION")
print("=" * 45)

# Calculate confusion matrix
cm = confusion_matrix(y_test, y_pred)
labels = ['FAKE', 'REAL']

# Interactive confusion matrix dengan Plotly
fig = px.imshow(
    cm,
    text_auto=True,
    aspect="auto",
    title="🔍 Confusion Matrix - BERT Fake News Detection",
    labels=dict(x="Predicted Label", y="Actual Label", color="Count"),
    x=labels,
    y=labels,
    color_continuous_scale='Blues'
)

fig.update_traces(
    texttemplate="%{z}",
    textfont_size=20,
    hovertemplate='Actual: %{y}<br>Predicted: %{x}<br>Count: %{z}<extra></extra>'
)

fig.update_layout(
    width=600,
    height=500,
    title_x=0.5,
    font_size=14
)

fig.show()

# Detailed confusion matrix analysis
print(f"📊 Confusion Matrix Analysis:")
tn, fp, fn, tp = cm.ravel()

print(f"   True Negatives (FAKE → FAKE): {tn}")
print(f"   False Positives (FAKE → REAL): {fp}")
print(f"   False Negatives (REAL → FAKE): {fn}")
print(f"   True Positives (REAL → REAL): {tp}")

# Calculate additional metrics
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0  # Recall for REAL
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0  # Recall for FAKE
precision_real = tp / (tp + fp) if (tp + fp) > 0 else 0
precision_fake = tn / (tn + fn) if (tn + fn) > 0 else 0

print(f"\n🎯 Advanced Metrics:")
print(f"   Sensitivity (REAL detection): {sensitivity:.4f}")
print(f"   Specificity (FAKE detection): {specificity:.4f}")
print(f"   REAL precision: {precision_real:.4f}")
print(f"   FAKE precision: {precision_fake:.4f}")

# Create detailed results dataframe
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
print(f"\n📋 Confusion Matrix DataFrame:")
display(cm_df)

🔍 CONFUSION MATRIX VISUALIZATION


📊 Confusion Matrix Analysis:
   True Negatives (FAKE → FAKE): 15
   False Positives (FAKE → REAL): 5
   False Negatives (REAL → FAKE): 1
   True Positives (REAL → REAL): 19

🎯 Advanced Metrics:
   Sensitivity (REAL detection): 0.9500
   Specificity (FAKE detection): 0.7500
   REAL precision: 0.7917
   FAKE precision: 0.9375

📋 Confusion Matrix DataFrame:


,FAKE,REAL
FAKE,15,5
REAL,1,19


In [ ]:
# Fungsi prediksi untuk berita baru
print("🧪 PREDICTION FUNCTION & TESTING")
print("=" * 45)

def predict_news(text, model, tokenizer, max_length=128):
    """
    Prediksi label berita (FAKE/REAL) untuk teks baru

    Args:
        text (str): Judul berita yang akan diprediksi
        model: Trained BERT model
        tokenizer: BERT tokenizer
        max_length (int): Maximum sequence length

    Returns:
        tuple: (predicted_label, confidence_score)
    """
    # Set model to evaluation mode
    model.eval()

    # Tokenize input
    inputs = tokenizer(
        text,
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors="pt"
    )

    # Make prediction
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
        predicted_class = torch.argmax(predictions, dim=-1).item()
        confidence = predictions[0][predicted_class].item()

    # Convert to label
    label = "REAL" if predicted_class == 1 else "FAKE"

    return label, confidence

# Test dengan contoh berita
print("🧪 Testing with sample news:")
print("-" * 40)

# Contoh 1: Berita yang terlihat FAKE
test_cases = [
    {
        "text": "SALAH Video Warga Israel Mengungsi di Gurun Pasir",
        "expected": "FAKE"
    },
    {
        "text": "PENIPUAN Tautan Pendaftaran CPNS Kemenhub",
        "expected": "FAKE"
    },
    {
        "text": "HOAX: Video viral tsunami di Jakarta ternyata rekayasa!",
        "expected": "FAKE"
    },
    {
        "text": "Bank Indonesia mempertahankan suku bunga acuan 5,75 persen",
        "expected": "REAL"
    },
    {
        "text": "Mayat Perempuan Ditemukan di Cisauk dalam Kondisi Tangan Terborgol",
        "expected": "REAL"
    }
]

# Test setiap case
test_results = []
for i, case in enumerate(test_cases, 1):
    prediction, confidence = predict_news(case["text"], model, tokenizer)

    correct = "✅" if prediction == case["expected"] else "❌"

    print(f"\n{i}. Text: {case['text'][:80]}...")
    print(f"   Expected: {case['expected']}")
    print(f"   Predicted: {prediction} (Confidence: {confidence:.4f}) {correct}")

    test_results.append({
        "Test_Case": i,
        "Text": case["text"][:50] + "...",
        "Expected": case["expected"],
        "Predicted": prediction,
        "Confidence": confidence,
        "Correct": prediction == case["expected"]
    })

# Create test results dataframe
test_df = pd.DataFrame(test_results)
print(f"\n📋 Test Results DataFrame:")
display(test_df)

# Calculate test accuracy
test_accuracy = sum(test_results[i]["Correct"] for i in range(len(test_results))) / len(test_results)
print(f"\n🎯 Test Cases Accuracy: {test_accuracy:.2%}")

🧪 PREDICTION FUNCTION & TESTING
🧪 Testing with sample news:
----------------------------------------

1. Text: SALAH Video Warga Israel Mengungsi di Gurun Pasir...
   Expected: FAKE
   Predicted: FAKE (Confidence: 0.8851) ✅

2. Text: PENIPUAN Tautan Pendaftaran CPNS Kemenhub...
   Expected: FAKE
   Predicted: FAKE (Confidence: 0.7614) ✅

3. Text: HOAX: Video viral tsunami di Jakarta ternyata rekayasa!...
   Expected: FAKE
   Predicted: FAKE (Confidence: 0.6550) ✅

4. Text: Bank Indonesia mempertahankan suku bunga acuan 5,75 persen...
   Expected: REAL
   Predicted: REAL (Confidence: 0.8022) ✅

5. Text: Mayat Perempuan Ditemukan di Cisauk dalam Kondisi Tangan Terborgol...
   Expected: REAL
   Predicted: REAL (Confidence: 0.7832) ✅

📋 Test Results DataFrame:


,Test_Case,Text,Expected,Predicted,Confidence,Correct
0,1,SALAH Video Warga Israel Mengungsi di Gurun Pa...,FAKE,FAKE,0.885089,True
1,2,PENIPUAN Tautan Pendaftaran CPNS Kemenhub...,FAKE,FAKE,0.761420,True
2,3,HOAX: Video viral tsunami di Jakarta ternyata ...,FAKE,FAKE,0.655012,True
3,4,Bank Indonesia mempertahankan suku bunga acuan...,REAL,REAL,0.802225,True
4,5,Mayat Perempuan Ditemukan di Cisauk dalam Kond...,REAL,REAL,0.783216,True



🎯 Test Cases Accuracy: 100.00%


In [ ]:
# Interface interaktif untuk prediksi manual
print("✏️ INTERACTIVE PREDICTION INTERFACE")
print("=" * 50)

def interactive_prediction():
    """
    Interface interaktif untuk testing model dengan input manual
    """
    print("🎯 BERT Fake News Detector - Interactive Mode")
    print("=" * 55)
    print("📝 Masukkan judul berita untuk diprediksi")
    print("💡 Tips: Coba dengan judul yang mengandung kata 'PENIPUAN', 'HOAX', 'GRATIS', dll")
    print("🚪 Ketik 'quit' untuk keluar")
    print("-" * 55)

    prediction_history = []

    while True:
        # Get user input
        user_input = input("\n📰 Masukkan judul berita: ").strip()

        # Check for exit condition
        if user_input.lower() in ['quit', 'exit', 'q', 'keluar']:
            print("👋 Terima kasih telah menggunakan BERT Fake News Detector!")
            break

        # Validate input
        if not user_input:
            print("❌ Input kosong! Silakan masukkan judul berita.")
            continue

        if len(user_input) < 10:
            print("⚠️ Judul terlalu pendek! Minimal 10 karakter.")
            continue

        # Make prediction
        try:
            prediction, confidence = predict_news(user_input, model, tokenizer)

            # Display results
            print(f"\n" + "="*60)
            print(f"📰 Input: {user_input}")
            print(f"🎯 Prediksi: {prediction}")
            print(f"📊 Confidence: {confidence:.4f} ({confidence*100:.1f}%)")

            # Interpretation
            if prediction == "FAKE":
                if confidence > 0.8:
                    interpretation = "🚨 SANGAT YAKIN ini berita palsu!"
                elif confidence > 0.6:
                    interpretation = "⚠️ Kemungkinan besar berita palsu"
                else:
                    interpretation = "🤔 Mungkin berita palsu (confidence rendah)"
                print(f"💭 {interpretation}")
            else:
                if confidence > 0.8:
                    interpretation = "✅ SANGAT YAKIN ini berita asli!"
                elif confidence > 0.6:
                    interpretation = "👍 Kemungkinan besar berita asli"
                else:
                    interpretation = "🤔 Mungkin berita asli (confidence rendah)"
                print(f"💭 {interpretation}")

            print("="*60)

            # Save to history
            prediction_history.append({
                "input": user_input,
                "prediction": prediction,
                "confidence": confidence
            })

        except Exception as e:
            print(f"❌ Error dalam prediksi: {str(e)}")
            print("💡 Coba dengan teks yang lebih pendek atau sederhana")

    # Show prediction history
    if prediction_history:
        print(f"\n📊 HISTORY PREDIKSI:")
        history_df = pd.DataFrame(prediction_history)
        history_df['input_short'] = history_df['input'].str[:50] + "..."
        display(history_df[['input_short', 'prediction', 'confidence']].round(4))

# Run interactive interface
interactive_prediction()

✏️ INTERACTIVE PREDICTION INTERFACE
🎯 BERT Fake News Detector - Interactive Mode
📝 Masukkan judul berita untuk diprediksi
💡 Tips: Coba dengan judul yang mengandung kata 'PENIPUAN', 'HOAX', 'GRATIS', dll
🚪 Ketik 'quit' untuk keluar
-------------------------------------------------------

📰 Masukkan judul berita: Jokowi Melakukan Penipuan

📰 Input: Jokowi Melakukan Penipuan
🎯 Prediksi: FAKE
📊 Confidence: 0.6144 (61.4%)
💭 ⚠️ Kemungkinan besar berita palsu

📰 Masukkan judul berita: exit
👋 Terima kasih telah menggunakan BERT Fake News Detector!

📊 HISTORY PREDIKSI:


,input_short,prediction,confidence
0,Jokowi Melakukan Penipuan...,FAKE,0.6144


In [ ]:
# Summary final dan save hasil
print("📋 FINAL MODEL SUMMARY & RESULTS")
print("=" * 50)

# Model performance summary
print("🎯 BERT FAKE NEWS DETECTION - FINAL RESULTS")
print("=" * 55)

# Dataset summary
print(f"📊 Dataset Summary:")
print(f"   Total articles processed: {len(df_balanced)}")
print(f"   Training samples: {len(X_train)}")
print(f"   Test samples: {len(X_test)}")
print(f"   Classes: FAKE (0) vs REAL (1)")
print(f"   Balance: 50-50 perfect split")

# Model summary
print(f"\n🤖 Model Summary:")
print(f"   Architecture: {model_name}")
print(f"   Task: Binary Text Classification")
print(f"   Input: News Headlines (max 128 tokens)")
print(f"   Output: FAKE/REAL prediction + confidence")
print(f"   Training epochs: {training_args.num_train_epochs}")
print(f"   Batch size: {training_args.per_device_train_batch_size}")

# Performance summary
print(f"\n📈 Performance Summary:")
print(f"   Test Accuracy: {accuracy:.4f} ({accuracy*100:.1f}%)")
print(f"   FAKE Precision: {class_report['FAKE']['precision']:.4f}")
print(f"   FAKE Recall: {class_report['FAKE']['recall']:.4f}")
print(f"   REAL Precision: {class_report['REAL']['precision']:.4f}")
print(f"   REAL Recall: {class_report['REAL']['recall']:.4f}")
print(f"   Macro F1-Score: {class_report['macro avg']['f1-score']:.4f}")

# Save hasil lengkap
results_summary = {
    'dataset_info': {
        'total_articles': len(df_balanced),
        'train_samples': len(X_train),
        'test_samples': len(X_test),
        'balance': '50-50 split'
    },
    'model_info': {
        'architecture': model_name,
        'max_length': 128,
        'epochs': training_args.num_train_epochs,
        'batch_size': training_args.per_device_train_batch_size
    },
    'performance': {
        'accuracy': float(accuracy),
        'fake_precision': float(class_report['FAKE']['precision']),
        'fake_recall': float(class_report['FAKE']['recall']),
        'real_precision': float(class_report['REAL']['precision']),
        'real_recall': float(class_report['REAL']['recall']),
        'macro_f1': float(class_report['macro avg']['f1-score'])
    }
}

# Save ke file
import json
with open('bert_results_summary.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

print(f"\n💾 Results saved to:")
print(f"   - bert_results_summary.json (performance metrics)")
print(f"   - balanced_dataset.csv (processed dataset)")

# Final recommendations
print(f"\n💡 RECOMMENDATIONS FOR RESEARCH:")
print(f"   ✅ Model successfully trained on Indonesian news headlines")
print(f"   ✅ Achieved {accuracy*100:.1f}% accuracy on balanced dataset")
print(f"   ✅ Ready for deployment and further testing")
print(f"   📝 Consider collecting more data for improved performance")
print(f"   📝 Test with real-world news headlines for validation")

print(f"\n🎉 BERT FAKE NEWS DETECTION PROJECT COMPLETED! 🎉")
print("🚀 Your model is ready for detecting fake news in Indonesian headlines!")

📋 FINAL MODEL SUMMARY & RESULTS
🎯 BERT FAKE NEWS DETECTION - FINAL RESULTS
📊 Dataset Summary:
   Total articles processed: 196
   Training samples: 156
   Test samples: 40
   Classes: FAKE (0) vs REAL (1)
   Balance: 50-50 perfect split

🤖 Model Summary:
   Architecture: indolem/indobert-base-uncased
   Task: Binary Text Classification
   Input: News Headlines (max 128 tokens)
   Output: FAKE/REAL prediction + confidence
   Training epochs: 3
   Batch size: 4

📈 Performance Summary:
   Test Accuracy: 0.8500 (85.0%)
   FAKE Precision: 0.9375
   FAKE Recall: 0.7500
   REAL Precision: 0.7917
   REAL Recall: 0.9500
   Macro F1-Score: 0.8485

💾 Results saved to:
   - bert_results_summary.json (performance metrics)
   - balanced_dataset.csv (processed dataset)

💡 RECOMMENDATIONS FOR RESEARCH:
   ✅ Model successfully trained on Indonesian news headlines
   ✅ Achieved 85.0% accuracy on balanced dataset
   ✅ Ready for deployment and further testing
   📝 Consider collecting more data for improve